# Creating and Deploying a Multi-Agent System with Genie and Serving Endpoints

This notebook builds a multi-agent system using Mosaic AI Agent Framework and [LangGraph](https://blog.langchain.dev/langgraph-multi-agent-workflows/), where [Genie](https://www.databricks.com/product/ai-bi/genie) is part of the agents.

In this notebook, we:
1. Author a multi-agent system using LangGraph.
1. Wrap the LangGraph agent with MLflow `ResponsesAgent` to ensure compatibility with Databricks features.
1. Manually test the multi-agent system's output.
1. Log and deploy the multi-agent system.

This example is based on [LangGraph documentation - Multi-agent supervisor example](https://github.com/langchain-ai/langgraph/blob/main/docs/docs/tutorials/multi_agent/agent_supervisor.md)

In [0]:
%pip install -U -qqq -U -qqq mlflow[databricks]
%pip install -U -qqq databricks-langchain
%pip install -U -qqq databricks-agents
%pip install -U -qqq uv
%pip install -U -qqqq databricks-langchain[memory] uv databricks-agents mlflow-skinny[databricks]

dbutils.library.restartPython()

### Initialization of Short term Memory

In [0]:
# First-time checkpoint table setup
from databricks.sdk import WorkspaceClient
from databricks_langchain import CheckpointSaver

# --- TODO: Fill in Lakebase instance name ---
INSTANCE_NAME = "runwai-postgres-database"

# Create tables if missing
with CheckpointSaver(instance_name=INSTANCE_NAME) as saver:
    saver.setup()           # sets up checkpoint tables
    print("✅ Checkpoint tables are ready.")

### Initialization of Long term Memory

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import DatabricksStore

# TODO: Fill in your Lakebase config values
LAKEBASE_INSTANCE_NAME = "runwai-postgres-database"
EMBEDDING_ENDPOINT = "databricks-qwen3-embedding-0-6b"
EMBEDDING_DIMS = 1024

store = DatabricksStore(
    instance_name=LAKEBASE_INSTANCE_NAME,
    embedding_endpoint=EMBEDDING_ENDPOINT,
    embedding_dims=EMBEDDING_DIMS,
    embedding_fields=["text", "fact"],
    )
store.setup()

### Semantic context retrieval

In [0]:
# Lecture des descriptions du Unity Catalog pour l'agent Builder Prompt
# Génère un fichier de contexte sémantique par persona (Stock, Cdistrib).

TABLES_STOCK = [
    "fshdh_poc_runwai.global_dc_flows_materials_and_components.ft_flow_components_detail",
    "fshdh_poc_runwai.global_dc_flows_materials_and_components.ft_flow_components_global",
    "fshdh_poc_runwai.global_dc_flows_materials_and_components.ft_stock_components",
    "fshdh_poc_runwai.component_referential.dm_base_component",
    "fshdh_poc_runwai.component_referential.dm_color_component",
    "fshdh_poc_runwai.component_referential.dm_component",
    "fshdh_poc_runwai.cost_prices_materials_and_components.ft_component_cost_price",
    "fshdh_poc_runwai.finished_product_referential.dm_product_sku",
    "fshdh_poc_runwai.finished_product_referential.dm_product_histo",
    "fshdh_poc_runwai.bills_of_materials.dm_bills_of_materials",
    "fshdh_poc_runwai.collection_referential.dm_collection",
    "fshdh_poc_runwai.components_global_buys_orders.ft_oa_line",
    "fshdh_poc_runwai.components_global_buys_orders.dm_oa_header",
    "fshdh_poc_runwai.fabrication_orders.ft_of_line",
    "fshdh_poc_runwai.fabrication_orders.dm_of_header",
    "fshdh_poc_runwai.label_tables.types_mouvement_orli"
]

TABLES_CDISTRIB = [
    "fshdh_poc_runwai.fashion_icone.ft_global_kpi_leadtime",
    "fshdh_poc_runwai.global_dc_activity_finished_products.dm_calendar_gdc",
    "fshdh_poc_runwai.global_dc_activity_finished_products.dm_epc",
    "fshdh_poc_runwai.global_dc_activity_finished_products.dm_product_ma",
    "fshdh_poc_runwai.global_dc_activity_finished_products.dm_sla_gdc_leadtime",
    "fshdh_poc_runwai.global_dc_activity_finished_products.ft_cq_ongoing",
    "fshdh_poc_runwai.global_dc_activity_finished_products.ft_cq_realized",
    "fshdh_poc_runwai.global_dc_activity_finished_products.ft_expedition",
    "fshdh_poc_runwai.global_dc_activity_finished_products.ft_kpi_detail_epc",
    "fshdh_poc_runwai.global_dc_activity_finished_products.ft_reception",
]

# Clé = identifiant persona (envoyé par le front en custom_inputs["persona"])
PERSONA_TABLES = {
    "Stock": TABLES_STOCK,
    "Cdistrib": TABLES_CDISTRIB,
}

SAMPLE_ROWS_LIMIT = 1
SEMANTIC_CONTEXT_DIR = "/tmp"

def _safe_str(v):
    if v is None:
        return ""
    s = str(v).strip()
    return s[:80] + "..." if len(s) > 80 else s

for persona_id, tables in PERSONA_TABLES.items():
    sections = []
    for table in tables:
        try:
            table_desc = ""
            try:
                ext = spark.sql(f"DESCRIBE TABLE EXTENDED {table}").collect()
                for row in ext:
                    if getattr(row, "col_name", None) == "Comment":
                        raw = getattr(row, "data_type", "") or ""
                        table_desc = str(raw).strip()
                        break
            except Exception:
                pass
            col_lines = []
            try:
                for row in spark.sql(f"DESCRIBE TABLE {table}").collect():
                    cname = getattr(row, "col_name", None)
                    if not cname or str(cname).startswith("#"):
                        continue
                    ctype = getattr(row, "data_type", "") or ""
                    ccomment = getattr(row, "comment", None) or ""
                    full_comment = str(ccomment).strip() if ccomment is not None else ""
                    col_lines.append(f"  - {cname} ({ctype}): {full_comment}")
            except Exception:
                col_lines = ["  (columns not available)"]
            sample_str = ""
            try:
                df = spark.sql(f"SELECT * FROM {table} LIMIT {SAMPLE_ROWS_LIMIT}")
                rows = df.collect()
                if rows:
                    cols = df.columns
                    sample_str = "  Exemple (" + str(len(rows)) + " ligne(s)):\n"
                    sample_str += "  " + " | ".join(cols) + "\n"
                    for r in rows:
                        vals = [_safe_str(r[c]) for c in cols]
                        sample_str += "  " + " | ".join(vals) + "\n"
                else:
                    sample_str = "  (aucune ligne)\n"
            except Exception as e:
                sample_str = "  (exemple non disponible: " + _safe_str(str(e)) + ")\n"
            block = f"### {table}\n{table_desc}\n" + "\n".join(col_lines) + "\n" + sample_str
            sections.append(block)
        except Exception as e:
            sections.append(f"### {table}\n(erreur: {_safe_str(str(e))})\n")

    path = f"{SEMANTIC_CONTEXT_DIR}/semantic_context_{persona_id.lower()}.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sections))
    print(f"Written {len(sections)} table(s) to {path} ({sum(len(s) for s in sections)} chars)")

### RAG Indexing Scheme
Fills the store with semantic chunks, so that in Advanced mode the `rag_schema` node retrieves only the tables relevant to the query

In [0]:
# Indexation RAG schema : remplir le store avec les chunks sémantiques 
import re

SEMANTIC_CONTEXT_DIR = "/tmp"
PERSONAS_RAG = ["stock", "cdistrib"]

def _chunk_semantic_file(content: str):
    """Découpe le fichier sémantique en blocs par table (### table_name)."""
    blocks = re.split(r"\n\s*###\s+", content.strip())
    out = []
    for b in blocks:
        b = b.strip()
        if not b:
            continue
        if not b.startswith("###"):
            b = "### " + b
        first_line = b.split("\n")[0]
        key = first_line.replace("###", "").strip()
        out.append((key or "unknown", b))
    return out

for persona in PERSONAS_RAG:
    path = f"{SEMANTIC_CONTEXT_DIR}/semantic_context_{persona}.txt"
    try:
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
    except FileNotFoundError:
        print(f"Fichier non trouvé: {path} — exécuter d'abord la cellule qui génère les fichiers sémantiques.")
        continue
    chunks = _chunk_semantic_file(content)
    namespace = ("schema", persona)
    for key, text in chunks:
        try:
            store.put(namespace, key, {"text": text})
        except Exception as e:
            print(f"Erreur put {persona}/{key}: {e}")
    print(f"Indexé {len(chunks)} chunks pour persona '{persona}' (namespace {namespace}).")

### Definition of the multi-agent system

In [0]:
%%writefile agent.py
import json
import operator
from typing import Annotated, Generator
import uuid
from typing import Any, Optional

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    CheckpointSaver,
    DatabricksFunctionClient,
    DatabricksStore,
    UCFunctionToolkit,
    set_uc_function_client,
)
from databricks_langchain.genie import GenieAgent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Send
from langgraph.config import get_stream_writer
import time
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from pydantic import BaseModel

client = DatabricksFunctionClient()
set_uc_function_client(client)

############################################
# Lakebase configuration
############################################

LAKEBASE_INSTANCE_NAME = "runwai-postgres-database"

# Long-term memory: embedding endpoint for semantic search
EMBEDDING_ENDPOINT = "databricks-qwen3-embedding-0-6b"
EMBEDDING_DIMS = 1024

########################################
# Models
########################################

GENIE = "genie"

class Genie(BaseModel):
    space_id: str
    name: str
    task: str = GENIE
    description: str

TOOLS = []

class AgentState(MessagesState):
    original_question: str
    enriched_instructions: str
    sub_questions: list[str]
    sub_results: Annotated[list[dict], operator.add]
    suggested_questions: Annotated[list[str], operator.add]
    thinking_mode: str = "Normal"  # "Normal" | "Approfondi"
    long_term_memory_context: Optional[str] = None  # Injected from DatabricksStore search (user_id)
    genie_simple_output: Optional[str] = None  # Raw Genie output (insight + SQL + table) for simple path
    custom_instructions: Optional[str] = None  # User preferences for style/tone (injected in response nodes only)
    response_level: str = "exploratoire"  # "exploratoire" | "statistique" 


########################################
# Prompts
########################################

INSTRUCTION_BUILDER_PROMPT = """Tu es un expert en analyse de questions métier sur des données logistiques.

Ta mission : produire des **instructions structurées** pour Genie (moteur text-to-SQL) afin qu'il génère la requête SQL optimale. 

Tu NE génères PAS de SQL. Tu NE proposes PAS de jointures.

## Ce que tu DOIS faire

1. **Identifier les tables et colonnes pertinentes** pour la question à partir du modèle de données ci-dessous (sous les instructions), et les mentionner à Genie pour qu'il sache où chercher.

2. **Rappeler les règles pertinentes** selon le type de question. Utilise UNIQUEMENT les règles suivantes (adapte au contexte, ne cite que ce qui s'applique) :

   2.1 **Règles d'agrégation** : 
   - Lorsque la question porte sur plusieurs entités (par exemple, « quels X », « quels Y »), vous DEVEZ agréger ces entités afin d'éviter les doublons.
   - Utilisez GROUP BY sur l'identifiant de l'entité afin de renvoyer une ligne par entité.
   - Fournissez des mesures agrégées utiles : COUNT(*) pour les comptes, MIN/MAX pour les dates/heures, SUM pour les quantités/montants.
   - Ne répertoriez PAS toutes les occurrences individuelles si la question porte sur des entités uniques.
   - La sélection finale SELECT doit renvoyer une ligne par entité avec des mesures agrégées.
   
   2.2 **Règles logiques** : 
   - Lorsque vous vérifiez les conditions sur les occurrences (par exemple, « éléments sans X », « enregistrements qui Y »), vous DEVEZ :
      a. Identifier d'abord TOUTES les occurrences qui pourraient correspondre à la condition (pas seulement la première/dernière à l'aide de MIN/MAX)
      b. Vérifier ensuite chaque occurrence individuellement par rapport à la condition
      c. Filtrer pour ne conserver que les occurrences qui correspondent à la condition
      d. Enfin, agréger les résultats filtrés par entité (GROUP BY).
   - N'utilisez PAS MIN() ou MAX() pour trouver une seule occurrence et ne vérifier que celle-ci, car cela vous ferait manquer d'autres occurrences qui correspondent également à la condition.
   - Le modèle correct : Identifier toutes les occurrences → Vérifier chacune d'elles → Filtrer celles qui correspondent → Agrégation par entité.
   - Vérifiez toujours : « Est-ce que je vérifie TOUTES les occurrences ou seulement une (MIN/MAX) ? »
   
   2.3 **Règles sur les critères multiples** : 
   - Lorsque la question mentionne plusieurs critères, TOUS les critères doivent être satisfaits simultanément - utilisez des conditions ET, et non OU
   - Lorsque vous comparez des entités au sein de groupes/catégories, utilisez des seuils spécifiques au groupe (percentiles ou moyennes au sein de chaque groupe), et non des seuils globaux
   
   2.4 **Règles sur la déviation implicite** : Lorsqu'une question suggère implicitement que certaines entités se comportent de manière significativement différente des autres au sein d'un même groupe, même si le mot « déviation » n'est pas explicitement utilisé, vous DEVEZ traiter cela comme une demande d'analyse de déviation. Les signaux typiques comprennent (liste non exhaustive) :
   - déséquilibre entre fréquence et intensité (rare mais important, peu fréquent mais percutant)
   - expressions de disproportion (« beaucoup plus que les autres », « peu mais très »)
   - effets de concentration (un petit nombre d'entités expliquent une part importante)
   - références à un comportement atypique, inhabituel ou non représentatif
   - comparaison implicite avec une norme de groupe ou un comportement « standard »
   Dans de tels cas, vous DEVEZ :
   a. Identifier la ou les mesures pertinentes par entité qui reflètent le comportement
   b. Calculer les statistiques de référence au niveau du groupe (au minimum la moyenne ; les percentiles seuls ne sont pas suffisants)
   c. Mesurer l'écart de chaque entité par rapport à la référence du groupe (z-score, pourcentage de la moyenne du groupe, rang centile ou équivalent)
   d. Appliquer des filtres de robustesse pour exclure les cas statistiquement peu fiables ou peu significatifs. Les seuils de robustesse par défaut doivent être prudents plutôt que permissifs
   
   2.5 **Règles d'analyse statistique**
   - Distinguez clairement entre :
        * Les critères de sélection (utilisés pour FILTRER les entités)
        * Les mesures de classement ou de priorisation (utilisées uniquement pour des ORDER BY ou à des fins descriptives)
   - N'utilisez PAS les mesures de classement (par exemple, le z-score) comme filtres implicites, sauf indication contraire explicite.
   - Lorsque vous comparez des entités au sein de groupes/familles :
        * Calculez toujours les statistiques spécifiques au groupe (moyenne, écart type, percentiles) au sein de chaque groupe à l'aide de GROUP BY.
        * Comparez la valeur de chaque entité aux statistiques de son propre groupe, et NON aux statistiques globales.
        * Utilisez des mesures relatives (z-score, percentile, pourcentage de la moyenne du groupe) pour une comparaison équitable entre les groupes.
  - Lors du calcul de ratios (par exemple, somme/nombre, total/occurrences) :
        * Calculez d'abord le ratio pour chaque entité.
        * Calculez ensuite les statistiques du groupe sur ces ratios.
        * Comparez le ratio de chaque entité aux statistiques de ratio de son groupe.

   2.6 **Règles de robustesse**: Lorsque vous effectuez une analyse de déviation ou des comparaisons statistiques, vous DEVEZ appliquer au moins un filtre de robustesse afin d'exclure les cas statistiquement peu fiables ou peu significatifs, tels que :
   - nombre minimum d'occurrences par entité (afin de garantir une taille d'échantillon suffisante)
   - quantité ou ampleur totale minimum (afin de garantir une échelle significative)
   - taille minimum du groupe (afin de garantir des statistiques de groupe fiables)
   - couverture ou exhaustivité minimum des données (afin de garantir des données représentatives)
   
3. **Préciser l'interprétation de la question** : Énumère les choix d'interprétation qu'il faut trancher pour coller au plus près à ce que l'utilisateur attend, et apporte une réponse pour chacune d'entre eux. Par exemple :
   - L'utilisateur veut-il des **entités uniques** (une ligne par composant) ou **toutes les occurrences** (chaque ligne de mouvement) ?
   - Quels sont les **critères explicites** à respecter ? Les termes comme "élevé", "faible" nécessitent-ils des seuils (percentiles, moyenne) ?
   - La question implique-t-elle une **analyse de déviation** ou une simple sélection ?
   - Les critères doivent-ils être combinés en **AND** ou en **OR** ?
   - Faut-il une **agrégation** (total, moyenne, décompte) ou une **liste brute** ?

4. ATTENTION : Privilégier les requêtes qui agrègent (GROUP BY, COUNT, SUM, AVG) ou qui limitent à un top N pertinent (ORDER BY … LIMIT N), pour que le résultat tienne en un nombre raisonnable de lignes tout en restant exhaustif sur la question posée. Si la question implique une liste potentiellement très longue, répondre par un résumé agrégé ou un top N, et indiquer le total (ex. 'Top 50 sur 1234 au total')

5. Si la dernière question fait référence à une question ou une réponse précédente (“ce mouvement”, “celui-là”, “à quelle heure”, etc.), utilise l’historique pour résoudre la référence.

## Ce que tu NE dois PAS faire

- Ne pas écrire de SQL
- Ne pas proposer de jointures ("JOIN X avec Y")
- Ne pas inventer des colonnes qui ne sont pas dans le modèle

## Format de sortie

Réponds en 1 à 2 paragraphes concis, dans la même langue que la question. Structure : tables/colonnes pertinentes identifiées, puis règles applicables, puis précisions d'interprétation.

## Modèle de données disponible (tables, colonnes, exemples de valeurs)

{semantic_context}

"""

DECOMPOSER_PROMPT = """Tu es un expert en décomposition de questions analytiques sur des données logistiques.

Tu reçois une question utilisateur et des instructions enrichies (contexte sémantique, tables/colonnes identifiées, règles applicables).

Ta mission : déterminer si la question nécessite une seule requête SQL ou plusieurs requêtes complémentaires.

RÈGLES DE DÉCOMPOSITION :
- Si la question est SIMPLE (une seule métrique, un seul angle d'analyse), retourne UNE seule sous-question identique à la question originale
- Si la question est COMPLEXE (plusieurs KPI, comparaisons, niveaux d'analyse différents, ou différentes analyses statistiques), décompose en 2-4 sous-questions INDÉPENDANTES
- Chaque sous-question doit pouvoir être traitée par une requête SQL indépendante
- Ne décompose PAS si les aspects sont interdépendants (ex: "le ratio entre X et Y" → une seule query)
- Pour chaque sous-question, privilégier une formulation qui amène une requête agrégée ou un top N (ex. 'Top 20 des…', 'Nombre de… par…'), sauf si l’utilisateur demande explicitement une liste détaillée.
- Maximum 4 sous-questions

RÈGLE CRITIQUE — ENRICHISSEMENT DES SOUS-QUESTIONS :
Chaque sous-question DOIT être auto-suffisante et détaillée. Elle DOIT reprendre explicitement les instructions que tu reçois :
1. Les TABLES et COLONNES spécifiques nécessaires (noms exacts tirés des instructions enrichies)
2. Les RÈGLES applicables à cette sous-question (agrégation, déviation, robustesse, etc.) ainsi que les métriques associées (si applicable)
3. Les PRÉCISIONS D'INTERPRÉTATION pertinentes (seuils, filtres, jointures logiques)

RÈGLE CRITIQUE - INDEPENDANCE DES SOUS-QUESTIONS:
Chaque sous-question doit être indépendante : exécutable sans le résultat des autres. Si une sous-question nécessite le résultat d’une autre (ex. “comparer à la moyenne du groupe” qui suppose d’abord “calculer la moyenne par groupe”), ne pas décomposer : retourner une seule sous-question ; Genie pourra utiliser des CTE/sous-requêtes.
* Mauvais : (1) métriques par composant, (2) stats par famille, (3) composants atypiques vs leur famille → la 3e dépend des 1 et 2.
* Bon : (1) par jour de la semaine, (2) par date, (3) par site → trois axes indépendants.

RÈGLE CRITIQUE - FORMAT DE SORTIE
Tu NE dois PAS écrire de SQL, ni proposer de jointures, ni décrire la structure de la requête (pas de GROUP BY, CTE, sous-requête, etc.).
Tu décris CE QUE tu veux obtenir, pas COMMENT le calculer.

MAUVAIS EXEMPLE (trop technique) :
  ❌ "SELECT fky_component, COUNT(cod_movement_number), AVG(qty_movement_quantity) FROM ft_flow_components_global GROUP BY fky_component HAVING COUNT(*) < (SELECT AVG(...) ...)"

MAUVAIS EXEMPLE (trop vague) :
  ❌ "Calculer les métriques par composant"

BON EXEMPLE :
  ✅ "Pour chaque composant (fky_component) dans ft_flow_components_global, je veux connaître le nombre de mouvements (basé sur cod_movement_number) et la quantité moyenne par mouvement (basée sur qty_movement_quantity). Associer chaque composant à sa famille matière (cod_fabric_family dans dm_component). Exclure les composants avec moins de 2 mouvements pour la robustesse statistique. Retourner une ligne par composant."

Format de sortie JSON strict :
{
  "sub_questions": ["sous-question enrichie 1", "sous-question enrichie 2"],
  "explanation": "Brève explication de la décomposition"
}

Réponds UNIQUEMENT en JSON valide. Formule les sous-questions dans la même langue que la question originale."""


SYNTHESIZER_PROMPT = """Tu es un analyste de données expert pour la Maison XXXXXXXX (M&C - Matières & Composants).

Tu reçois les résultats de plusieurs sous-requêtes SQL exécutées par Genie sur les données.
Chaque résultat comprend la sous-question posée et la réponse de Genie (interprétation, SQL, tableau de résultats).

Ta mission : synthétiser tous ces résultats en UNE SEULE réponse cohérente et structurée.

RÈGLES NON NÉGOCIABLES :
- Formule une réponse intégrée, pas une simple juxtaposition des résultats
- Inclue toujours les valeurs numériques exactes des résultats
- Si des métriques ou des méthodes particulières ont été utilisées dans l'analyse, toujours le dire explicitement et le justifier
- Si les résultats se contredisent ou sont incomplets, le signaler
- Structure avec des sections si plusieurs aspects sont couverts
- Lorsque l'ensemble de résultats est petit (10 lignes ou moins), liste toutes les entités explicitement
- Lorsque l'ensemble de résultats est volumineux, fournir d’abord le total (count ou agrégat) puis un extrait limité (top N ou LIMIT) en précisant clairement que c’est un extrait
- Sois concis et précis
- Ne met pas d'emoji dans tes réponses"""


MEMORY_SAVE_PROMPT = """Analyse le dernier message de l'utilisateur. A-t-il partagé une préférence, un fait important à retenir, ou demandé d'oublier quelque chose ?

Si OUI (fait/préférence à retenir), retourne : {"action": "save", "key": "clé_descriptive_courte", "data": {"fact": "le fait à retenir"}}
Si l'utilisateur demande d'oublier un élément précis : {"action": "delete", "key": "clé_exacte_existante"}
Si l'utilisateur demande de tout oublier / tout supprimer : {"action": "delete_all"}
Si RIEN à retenir (question analytique, requête de données, etc.) : {"action": "none"}

RÈGLES pour la suppression :
- Pour "delete", utilise EXACTEMENT une des clés listées dans les mémoires existantes (entre crochets).
- Si la clé demandée n'existe pas dans les mémoires existantes, retourne {"action": "none"}.
- Ne devine PAS de clé : utilise uniquement celles fournies.

Ne sauvegarde PAS les détails transitoires, les questions analytiques, ou les requêtes de données.
Réponds UNIQUEMENT en JSON valide, sans explication."""


# Addendum ajouté aux prompts quand response_level == "statistique"
INSTRUCTION_BUILDER_STAT_ADDENDUM = """
[MODE STATISTIQUE] Propose des analyses statistiques (écarts-types, percentiles, z-scores, comparaisons au groupe, robustesse) uniquement si la question le justifie (comparaisons, écarts, benchmarks, positionnement par rapport à un groupe). Si la question est purement factuelle ou descriptive, ne pas imposer de couche statistique.
.
"""

DECOMPOSER_STAT_ADDENDUM = """
[MODE STATISTIQUE] N’inclure des analyses statistiques dans les sous-questions que lorsque la question originale ou les instructions enrichies le justifient. Ne pas forcer des stats sur des questions simples ou descriptives.
"""

SYNTHESIZER_STAT_ADDENDUM = """
[MODE STATISTIQUE] Mets en avant les méthodes statistiques lorsqu’elles ont effectivement été utilisées pour répondre à la question. Ne pas inventer ou surligner des stats si la réponse n’en contient pas. Justifie les choix statistiques.
"""

GENIE_STAT_PREFIX = (
    "[MODE STATISTIQUE] : Utilise des analyses statistiques (écarts-types, percentiles, z-scores, comparaisons au groupe) seulement si la question le requiert (comparaisons, écarts, benchmarks). Sinon, réponds de façon directe sans forcer de métriques statistiques."
)

########################################
# Agent Graph 
########################################

def create_agent_graph(
    llm: Runnable,
    genie_config: Genie,
    semantic_context: str = "",
    checkpointer: Optional[Any] = None,
    store: Optional[Any] = None,
    llm_fast: Optional[Runnable] = None,
):
    instruction_prompt = INSTRUCTION_BUILDER_PROMPT.format(semantic_context=semantic_context)

    def worker_message_processor(messages):
        """Dernière question + contexte de la conversation (mémoire court terme) pour Genie."""
        human_contents = []
        last_assistant_content = ""
        for msg in messages:
            typ = getattr(msg, "type", None)
            content = getattr(msg, "content", "") or ""
            content = content if isinstance(content, str) else str(content)
            if not content.strip():
                continue
            if typ == "human":
                human_contents.append(content)
            else:
                # Garder la dernière réponse assistant 
                last_assistant_content = content

        if not human_contents:
            return ""
        latest_question = human_contents[-1]

        # Une seule question → pas de contexte
        if len(human_contents) <= 1:
            return latest_question

        # Suivi : ajouter le contexte pour que Genie comprenne "il", "ce mouvement", etc.
        context = ""
        if last_assistant_content:
            context = (
                "Contexte de la conversation (réponse précédente, à utiliser pour les références comme 'il', 'ce mouvement', 'à quelle heure'):\n"
                f"{last_assistant_content[:4000]}{'...' if len(last_assistant_content) > 4000 else ''}\n\n"
            )
        # Inclure la dernière question utilisateur pour les suivis ("Et à quelle heure ?", etc.)
        if len(human_contents) >= 2:
            prev_question = human_contents[-2]
            context += f"Dernière question :\n{prev_question[:2000]}{'...' if len(prev_question) > 2000 else ''}\n\n"
        return context + "Question actuelle:\n" + latest_question

    worker_genie = GenieAgent(
        genie_space_id=genie_config.space_id,
        genie_agent_name=genie_config.name,
        description=genie_config.description,
        message_processor=worker_message_processor,
        include_context=True,
    )

    # Patch Genie.ask_question pour exposer message_id et utiliser conversation_id (mémoire native Genie)
    _genie = worker_genie.func.keywords["genie"]
    _original_ask = _genie.ask_question
    def _ask_with_message_id(question, conversation_id=None):
        cid = getattr(_genie, "_next_conversation_id", None) or conversation_id
        if getattr(_genie, "_next_conversation_id", None):
            _genie._next_conversation_id = None
        if not cid:
            resp = _genie.start_conversation(question)
        else:
            resp = _genie.create_message(cid, question)
        if not hasattr(_genie, "_last_message_ids"):
            _genie._last_message_ids = {}
        _genie._last_message_ids[resp.get("conversation_id", "")] = resp.get("message_id")

        genie_response = _genie.poll_for_result(resp["conversation_id"], resp["message_id"])
        if not genie_response.conversation_id:
            genie_response.conversation_id = resp["conversation_id"]
        return genie_response
    _genie.ask_question = _ask_with_message_id

    def _fetch_genie_message_data(genie, space_id, conversation_id, message_id):
        """Récupère la partie textuelle (insight), suggested_questions et is_truncated via GET message puis GET attachment query result."""
        path = f"/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}"
        data = genie.genie._api.do("GET", path, headers=genie.headers)
        parts = []
        all_questions = []
        for att in data.get("attachments", []):
            content = (att.get("text") or {}).get("content", "").strip()
            if content:
                parts.append(content)
            suggested = att.get("suggested_questions")
            if suggested and isinstance(suggested.get("questions"), list):
                for q in suggested["questions"]:
                    if isinstance(q, str) and q.strip():
                        all_questions.append(q.strip())
        is_truncated = None
        attachment_id = None
        for att in data.get("attachments", []):
            att = att if isinstance(att, dict) else (getattr(att, "as_dict", None) and att.as_dict() or vars(att))
            if isinstance(att, dict) and att.get("query") and att.get("attachment_id"):
                attachment_id = att["attachment_id"]
                break
        if attachment_id:
            try:
                path_qr = f"/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}/attachments/{attachment_id}/query_result"
                qr_data = genie.genie._api.do("GET", path_qr, headers=genie.headers)
                manifest = (qr_data.get("statement_response") or {}).get("manifest")
                if isinstance(manifest, dict) and "truncated" in manifest:
                    is_truncated = manifest.get("truncated") is True
            except Exception:
                pass
        if is_truncated is None:
            query_result = data.get("query_result") or {}
            if query_result.get("row_count") == 5000:
                is_truncated = True
            else:
                for att in data.get("attachments", []):
                    att = att if isinstance(att, dict) else (getattr(att, "as_dict", None) and att.as_dict() or vars(att))
                    q = (att.get("query") or {}) if isinstance(att, dict) else {}
                    q = q if isinstance(q, dict) else (getattr(q, "as_dict", None) and q.as_dict() or vars(q))
                    meta = (q.get("query_result_metadata") or {}) if isinstance(q, dict) else {}
                    if isinstance(meta, dict) and meta.get("row_count") == 5000:
                        is_truncated = True
                        break
        return "\n\n".join(parts), all_questions, is_truncated

    GENIE_TRUNCATION_NOTE = (
        "**Note :** Les résultats du tableau sont tronqués à 5000 lignes (limite Genie). Préciser explicitement que les résultats sont tronqués."
    )

    def route_by_mode(state: AgentState):
        """Route : Approfondi → instruction_builder, sinon → simple_genie."""
        if state.get("thinking_mode") == "Approfondi":
            return "instruction_builder"
        return "simple_genie"
    
    # ---- Nodes ----
    def simple_genie(state: AgentState, config: RunnableConfig) -> dict:
        """Chemin Normal : question utilisateur → Genie → résultat brut (SQL + table). Utilise la mémoire native Genie si genie_conversation_id est fourni."""
        writer = get_stream_writer()
        if writer:
            writer({"type": "graph_step", "node": "simple_genie", "label": "Requête Genie", "emitted_at": time.time()})
        messages = list(state["messages"])

        last_human_idx = None
        orig_str = ""
        original_question = ""

        for i in range(len(messages) - 1, -1, -1):
            msg_type = getattr(messages[i], "type", None) or getattr(messages[i], "role", None)
            if msg_type in ("human", "user"):
                last_human_idx = i
                orig = getattr(messages[i], "content", "") or ""
                orig_str = orig if isinstance(orig, str) else str(orig)
                original_question = orig_str
                break

        long_term = (state.get("long_term_memory_context") or "").strip()
        response_level = state.get("response_level", "exploratoire")
        custom = (state.get("custom_instructions") or "").strip()

        # Contenu de la question (avec préfixes optionnels)
        content = orig_str
        if (long_term or response_level == "statistique" or custom) and last_human_idx is not None:
            prefixes = []
            if custom:
                prefixes.append(f"Instructions personnalisées (style, ton, format de réponse souhaité):\n{custom}")
            if long_term:
                prefixes.append(f"Mémoire long terme:\n{long_term}")
            if response_level == "statistique":
                prefixes.append(GENIE_STAT_PREFIX)
            content = "\n\n".join(prefixes) + "\n\nQuestion utilisateur:\n" + orig_str

        # Mémoire court terme : utiliser la conversation Genie native si on a un conversation_id
        genie_conv_id = (config or {}).get("configurable", {}).get("genie_conversation_id") if config else None
        if genie_conv_id:
            _genie = worker_genie.func.keywords["genie"]
            _genie._next_conversation_id = genie_conv_id
            messages_for_genie = [HumanMessage(content=content)]
        else:
            if (long_term or response_level == "statistique" or custom) and last_human_idx is not None:
                messages = (
                    messages[:last_human_idx]
                    + [HumanMessage(content=content)]
                    + messages[last_human_idx + 1:]
                )
            messages_for_genie = messages

        result = worker_genie.invoke({"messages": messages_for_genie}, config=config or {})
        result_msgs = result.get("messages", [])

        # Récupérer l'insight textuel Genie (GET message)
        genie_response_text = ""
        _genie = worker_genie.func.keywords["genie"]
        message_id = (getattr(_genie, "_last_message_ids", None) or {}).get(result.get("conversation_id"))
        
        suggested_questions = []
        is_truncated = None
        if result.get("conversation_id") and message_id:
            try:
                genie_response_text, suggested_questions, is_truncated = _fetch_genie_message_data(
                    _genie, genie_config.space_id, result["conversation_id"], message_id
                )
                if result.get("conversation_id") and getattr(_genie, "_last_message_ids", None):
                    _genie._last_message_ids.pop(result.get("conversation_id"), None)
            except Exception:
                pass
            if writer:
                writer({"type": "genie_message_ids", "conversation_id": result["conversation_id"], "message_id": message_id})
            if writer and suggested_questions:
                writer({"type": "suggested_questions", "suggested_questions": suggested_questions})
            if writer and is_truncated is not None and is_truncated:
                writer({"type": "genie_truncated", "is_truncated": True})

        # Extraire par type (reasoning, sql, result) puis ordre logique : reasoning → insight → SQL → tableau
        reasoning_content = ""
        sql_content = ""
        table_content = ""
        for msg in result_msgs:
            content = getattr(msg, "content", "") or ""
            if not (isinstance(content, str) and content.strip()):
                continue
            name = getattr(msg, "name", None)
            if name == "query_reasoning":
                reasoning_content = content.strip()
            elif name == "query_sql":
                sql_content = content.strip()
            elif name == "query_result":
                table_content = content.strip()

        genie_parts = []
        if reasoning_content:
            genie_parts.append(reasoning_content)
        if genie_response_text:
            genie_parts.append(genie_response_text.strip())
        if sql_content:
            genie_parts.append(sql_content)
        if table_content:
            genie_parts.append(table_content)
        if is_truncated is not None and is_truncated:
            genie_parts.append(GENIE_TRUNCATION_NOTE)
        genie_output = "\n\n".join(genie_parts)

        # Inclure l'insight en premier message pour le stream
        out_messages = list(result_msgs)
        if genie_response_text:
            out_messages.insert(0, AIMessage(content=genie_response_text, name="query_response_text"))

        return {
            "messages": out_messages,
            "original_question": original_question,
            "genie_simple_output": genie_output,
            "suggested_questions": suggested_questions,
        }


    def instruction_builder(state: AgentState) -> dict:
        """Enriches the user question with structured instructions for Genie."""
        writer = get_stream_writer()
        if writer:
            writer({"type": "graph_step", "node": "instruction_builder", "label": "Enrichissement de la question"})
        messages = state["messages"]
        last_content = messages[-1].content
        user_message = last_content if isinstance(last_content, str) else str(last_content)

        # Contexte conversationnel (mémoire court terme) : derniers échanges
        MAX_TURNS = 3  # ex. 3 derniers tours user/assistant
        conv_parts = []
        count = 0
        for msg in reversed(messages[:-1]):  # tout sauf le dernier (déjà la question courante)
            if count >= MAX_TURNS * 2:
                break
            role = "Utilisateur" if getattr(msg, "type", None) == "human" else "Assistant"
            content = getattr(msg, "content", "") or ""
            if isinstance(content, str) and content.strip():
                conv_parts.append(f"{role}: {content[:2000]}{'...' if len(content) > 2000 else ''}")
                count += 1
        conversation_context = "\n".join(reversed(conv_parts)) if conv_parts else ""

        long_term = (state.get("long_term_memory_context") or "").strip()
        memory_block = f"Mémoire long terme (informations enregistrées sur l'utilisateur):\n{long_term}\n\n" if long_term else ""

        prompt_content = memory_block + (
            (f"Historique récent de la conversation:\n{conversation_context}\n\n"
             f"Dernière question à traiter (répondre en tenant compte du contexte ci-dessus):\n{user_message}")
            if conversation_context else user_message
        )

        response_level = state.get("response_level", "exploratoire")
        system_content = instruction_prompt  # déjà défini
        if response_level == "statistique":
            system_content += f"\n\n{INSTRUCTION_BUILDER_STAT_ADDENDUM}"
        response = llm.invoke([
            SystemMessage(content=system_content),
            HumanMessage(content=prompt_content),
        ], temperature=0)
        return {
            "original_question": user_message,
            "enriched_instructions": response.content,
        }

    def decomposer(state: AgentState) -> dict:
        """Breaks a complex question into independent sub-questions."""
        writer = get_stream_writer()
        if writer:
            writer({"type": "graph_step", "node": "decomposer", "label": "Décomposition en sous-questions", "emitted_at": time.time()})
        original = state.get("original_question", "")
        enriched = state.get("enriched_instructions", "")
        
        response_level = state.get("response_level", "exploratoire")
        decomposer_prompt = DECOMPOSER_PROMPT
        if response_level == "statistique":
            decomposer_prompt += f"\n\n{DECOMPOSER_STAT_ADDENDUM}"
        response = llm.invoke([
            SystemMessage(content=decomposer_prompt),
            HumanMessage(
                content=f"Question originale :\n{original}\n\nInstructions enrichies :\n{enriched}"
                ),
            ],
            temperature=0,
        )

        try:
            text = response.content.strip()
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            parsed = json.loads(text.strip())
            sub_questions = parsed.get("sub_questions", [original])
        except (json.JSONDecodeError, KeyError):
            sub_questions = [original]

        if not sub_questions:
            sub_questions = [original]

        return {"sub_questions": sub_questions}

    def route_to_genies(state: AgentState):
        """Fan-out: creates one Send() per sub-question, each carrying the full enrichment."""
        enriched = state.get("enriched_instructions", "")
        sub_qs = state.get("sub_questions", [state.get("original_question", "")])

        return [
            Send("genie_worker", {
                "messages": [HumanMessage(
                    content=(
                        f"INSTRUCTIONS pour la requête SQL :\n{enriched}\n\n"
                        f"Question de l'utilisateur :\n{sq}"
                    )
                )],
                "sub_questions": [sq],
                "sub_results": [],
                "genie_worker_index":i+1,
                "genie_worker_total":len(sub_qs),
            })
            for i, sq in enumerate(sub_qs)
        ]

    MAX_GENIE_OUTPUT_CHARS = 200000

    def genie_worker(state: AgentState) -> dict:
        writer = get_stream_writer()
        if writer:
            i = state.get("genie_worker_index", 1)
            n = state.get("genie_worker_total", 1)
            label = f"Requête Genie ({i}/{n})" if n > 1 else "Requête Genie"
            writer({"type": "graph_step", "node": "genie_worker", "label": label, "emitted_at": time.time()})
        result = worker_genie.invoke({"messages": state["messages"]})
        result_msgs = result.get("messages", [])

        # Récupérer message_id (exposé par le patch) et insight textuel Genie
        _genie = worker_genie.func.keywords["genie"]
        message_id = (getattr(_genie, "_last_message_ids", None) or {}).get(result.get("conversation_id"))
        if message_id:
            result = dict(result)
            result["message_id"] = message_id
        genie_response_text = ""
        suggested_questions = []
        is_truncated = None
        if result.get("conversation_id") and result.get("message_id"):
            try:
                genie_response_text, suggested_questions, is_truncated = _fetch_genie_message_data(
                    _genie, genie_config.space_id, result["conversation_id"], result["message_id"]
                )
                if result.get("conversation_id") and getattr(_genie, "_last_message_ids", None):
                    _genie._last_message_ids.pop(result.get("conversation_id"), None)
            except Exception:
                pass
            if writer:
                writer({"type": "genie_message_ids", "conversation_id": result["conversation_id"], "message_id": result["message_id"]})
            if writer and is_truncated is not None and is_truncated:
                writer({"type": "genie_truncated", "is_truncated": True})

        # Extraire le message SQL pour l'affichage (audit)
        sql_msg = None
        result_msg = None
        for msg in result_msgs:
            name = getattr(msg, "name", None)
            if name == "query_sql" and getattr(msg, "content", None):
                sql_msg = msg
            if name == "query_result" and getattr(msg, "content", None):
                result_msg = msg  

        genie_parts = []
        if genie_response_text:
            genie_parts.append(genie_response_text)
        for msg in result_msgs:
            content = getattr(msg, "content", "")
            if content and isinstance(content, str) and content.strip():
                genie_parts.append(content)

        genie_output = "\n\n".join(genie_parts)

        if len(genie_output) > MAX_GENIE_OUTPUT_CHARS:
            data_lines = [l for l in genie_output.split("\n")
                        if l.strip().startswith("|") and not l.strip().startswith("|---")]
            total_data_lines = max(0, len(data_lines) - 1)

            truncated = genie_output[:MAX_GENIE_OUTPUT_CHARS]
            last_nl = truncated.rfind("\n")
            if last_nl > 0:
                truncated = truncated[:last_nl]

            genie_output = truncated + f"\n\n... ({total_data_lines} lignes au total, résultats tronqués)"
        if is_truncated is not None and is_truncated:
            genie_output = genie_output + "\n\n" + GENIE_TRUNCATION_NOTE

        sub_q = state.get("sub_questions", [""])[0] if state.get("sub_questions") else ""

        out = {
            "sub_results": [{
                "sub_question": sub_q,
                "genie_output": genie_output,
            }]
        }
        if sql_msg:
            out["messages"] = out.get("messages", []) + [sql_msg]
        if result_msg:
            out["messages"] = out.get("messages", []) + [result_msg]
        out["suggested_questions"] = suggested_questions
        return out

    def synthesizer(state: AgentState) -> dict:
        """Combines all Genie results into a single coherent response."""
        writer = get_stream_writer()
        if writer:
            writer({"type": "graph_step", "node": "synthesizer", "label": "Synthèse des résultats", "emitted_at": time.time()})
        sq = state.get("suggested_questions") or []
        if sq:
            seen = set()
            deduped = []
            for q in sq:
                if isinstance(q, str):
                    q = q.strip()
                    if q and q not in seen:
                        seen.add(q)
                        deduped.append(q)
            deduped = deduped[:3]
            if writer and deduped:
                writer({"type": "suggested_questions", "suggested_questions": deduped})
        original = state.get("original_question", "")
        sub_results = state.get("sub_results", [])

        results_text = []
        for r in sub_results:
            results_text.append(
                f"### Sous-question : {r.get('sub_question', 'N/A')}\n"
                f"{r.get('genie_output', 'Pas de résultat')}"
            )

        all_results = "\n\n---\n\n".join(results_text)

        custom = (state.get("custom_instructions") or "").strip()
        response_level = state.get("response_level", "exploratoire")
        system_content = (
            f"Instructions personnalisées de l'utilisateur (style, ton, comportement) :\n{custom}\n\n"
            + SYNTHESIZER_PROMPT
        ) if custom else SYNTHESIZER_PROMPT
        if response_level == "statistique":
            system_content += f"\n\n{SYNTHESIZER_STAT_ADDENDUM}"
            
        response = llm.invoke([
            SystemMessage(content=system_content),
            HumanMessage(
                content=(
                    f"Question originale : {original}\n\n"
                    f"Nombre de sous-requêtes : {len(sub_results)}\n\n"
                    f"Résultats :\n{all_results}"
                )
            ),
        ])
        return {"messages": [response]}

    # ---- Memory nodes (deterministic get + LLM-based save) ----

    def memory_get(state: AgentState, config: RunnableConfig) -> dict:
        """Deterministic memory lookup: semantic search without LLM calls."""
        if not store:
            return {"long_term_memory_context": None}
        user_id = config.get("configurable", {}).get("user_id")
        if not user_id:
            return {"long_term_memory_context": None}
        writer = get_stream_writer()
        if writer:
            writer({"type": "graph_step", "node": "memory_get", "label": "Récupération mémoire long terme", "emitted_at": time.time()})

        last_question = ""
        for msg in reversed(state["messages"]):
            if getattr(msg, "type", None) == "human":
                last_question = getattr(msg, "content", "") or ""
                if not isinstance(last_question, str):
                    last_question = str(last_question)
                break

        if not last_question:
            return {"long_term_memory_context": None}

        namespace = ("user_memories", user_id.replace(".", "-"))
        try:
            results = store.search(namespace, query=last_question, limit=5)
        except Exception:
            return {"long_term_memory_context": None}

        if not results:
            return {"long_term_memory_context": None}

        context = "\n".join(
            f"- [{item.key}]: {json.dumps(item.value)}" for item in results
        )
        return {"long_term_memory_context": context}

    def memory_save(state: AgentState, config: RunnableConfig) -> dict:
        """Post-response: LLM decides if user shared something worth saving."""
        if not store:
            return {}
        user_id = config.get("configurable", {}).get("user_id")
        if not user_id:
            return {}
    

        last_human = ""
        for msg in reversed(state["messages"]):
            if getattr(msg, "type", None) == "human":
                last_human = getattr(msg, "content", "") or ""
                if not isinstance(last_human, str):
                    last_human = str(last_human)
                break
        if not last_human:
            return {}

        memory_context = (state.get("long_term_memory_context") or "").strip()
        prompt = MEMORY_SAVE_PROMPT
        if memory_context:
            prompt += f"\n\nMémoires existantes de l'utilisateur :\n{memory_context}"
        else:
            prompt += "\n\nAucune mémoire existante pour cet utilisateur."

        save_llm = llm_fast if llm_fast else llm
        try:
            response = save_llm.invoke([
                SystemMessage(content=prompt),
                HumanMessage(content=f"Message utilisateur : {last_human}"),
            ], temperature=0)
            text = response.content.strip()
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            parsed = json.loads(text.strip())

            namespace = ("user_memories", user_id.replace(".", "-"))
            if parsed.get("action") == "save":
                store.put(namespace, parsed["key"], parsed["data"])
            elif parsed.get("action") == "delete":
                store.delete(namespace, parsed["key"])
            elif parsed.get("action") == "delete_all":
                try:
                    items = store.search(namespace, query="", limit=100)
                    for item in items:
                        store.delete(namespace, item.key)
                except Exception:
                    pass
        except (json.JSONDecodeError, KeyError, Exception):
            pass

        return {}

    # ---- Graph wiring ----

    graph = StateGraph(AgentState)
    graph.add_node("memory_get", memory_get)
    graph.add_node("simple_genie", simple_genie)
    graph.add_node("instruction_builder", instruction_builder)
    graph.add_node("decomposer", decomposer)
    graph.add_node("genie_worker", genie_worker)
    graph.add_node("synthesizer", synthesizer)
    graph.add_node("memory_save", memory_save)

    use_memory = store is not None

    if use_memory:
        graph.add_edge(START, "memory_get")
        graph.add_conditional_edges("memory_get", route_by_mode, ["simple_genie", "instruction_builder"])
    else:
        graph.add_conditional_edges(START, route_by_mode, ["simple_genie", "instruction_builder"])

    # Après simple_genie : direct memory/END (insight Genie affiché tel quel)
    graph.add_edge("simple_genie", "memory_save" if use_memory else END)

    # Mode Approfondi
    graph.add_edge("instruction_builder", "decomposer")
    graph.add_conditional_edges("decomposer", route_to_genies)
    graph.add_edge("genie_worker", "synthesizer")
    graph.add_edge("synthesizer", "memory_save" if use_memory else END)

    if use_memory:
        graph.add_edge("memory_save", END)

    if checkpointer is not None:
        return graph.compile(checkpointer=checkpointer)
    return graph.compile()


##########################################
# Wrap Sequential Graph as ResponsesAgent
##########################################

HIDDEN_NODES = {"instruction_builder", "decomposer", "memory_get", "memory_save"}

def _should_emit(msg) -> bool:
    """Filtre les messages de handoff et intermédiaires."""
    content = getattr(msg, "content", None)
    if not content:
        return False

    text = content if isinstance(content, str) else str(content)
    if not text.strip():
        return False

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        return False

    if getattr(msg, "type", None) == "tool":
        return False
    
    # Exclure les sorties Genie sauf query_sql et query_result
    if getattr(msg, "name", None)=="query_reasoning":
        return False

    # Exclure les sorties Genie si query_result sans tableau
    if getattr(msg, "name", None) == "query_result" and "|" not in text:
        return False
        
    return True


class LangGraphResponsesAgent(ResponsesAgent):
    """Supports multiple personas via custom_inputs['persona'] (e.g. 'Stock', 'Cdistrib')."""

    def __init__(self,
        lakebase_instance_name: str,
        llm: Runnable,
        personas: dict,
        default_persona: str = "Stock",
        llm_fast: Optional[Runnable] = None,
    ):
        self.lakebase_instance_name = lakebase_instance_name
        self.llm = llm
        self.llm_fast = llm_fast
        self.personas = personas  # persona_id -> (Genie, semantic_context: str)
        self.default_persona = default_persona
        self._store = None

    @property
    def store(self):
        """Lazy init of DatabricksStore with semantic search (long-term memory)."""
        if self._store is None:
            self._store = DatabricksStore(
                instance_name=self.lakebase_instance_name,
                embedding_endpoint=EMBEDDING_ENDPOINT,
                embedding_dims=EMBEDDING_DIMS,
            )
            self._store.setup()
        return self._store

    def _get_user_id(self, request: ResponsesAgentRequest) -> Optional[str]:
        """User id from chat context for long-term memory (same as reference)."""
        if request.context and getattr(request.context, "user_id", None):
            return request.context.user_id
        return (request.custom_inputs or {}).get("user_id")

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        ci = dict(request.custom_inputs or {})
        if "thread_id" in ci:
            return ci["thread_id"]
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        user_id = self._get_user_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        if user_id:
            ci["user_id"] = user_id
        request.custom_inputs = ci

        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        checkpoint_config = {"configurable": {"thread_id": thread_id}}
        if user_id:
            checkpoint_config["configurable"]["user_id"] = user_id
        genie_cid = ci.get("genie_conversation_id")
        if genie_cid:
            checkpoint_config["configurable"]["genie_conversation_id"] = genie_cid
        first_message = True
        seen_ids = set()
        accumulated_output_messages = []

        memory_store = self.store if user_id else None

        persona = (request.custom_inputs or {}).get("persona") or self.default_persona
        if persona not in self.personas:
            persona = self.default_persona
        genie_config, semantic_context = self.personas[persona]

        # IDs des messages d'entrée (à ne jamais ré-émettre)
        input_msg_ids = set()
        for m in cc_msgs:
            if hasattr(m, "id") and m.id:
                input_msg_ids.add(m.id)

        with CheckpointSaver(instance_name=self.lakebase_instance_name) as checkpointer:
            graph = create_agent_graph(
                self.llm,
                genie_config,
                semantic_context=semantic_context,
                checkpointer=checkpointer,
                store=memory_store,
                llm_fast=self.llm_fast,
            )

            thinking_mode = "Normal"
            custom_instructions = None
            response_level = "exploratoire"
            if getattr(request, "custom_inputs", None):
                ci = request.custom_inputs
                v = ci.get("thinking_mode", "Normal")
                if v is True:
                    thinking_mode = "Approfondi"
                elif v is False:
                    thinking_mode = "Normal"
                else:
                    thinking_mode = str(v) if v in ("Normal", "Approfondi") else "Normal"
                custom_instructions = ci.get("custom_instructions") or None
                response_level = ci.get("response_level", "exploratoire")
            
            initial_state= {
                "messages": cc_msgs,
                "thinking_mode": thinking_mode,
                "response_level": response_level,
                "long_term_memory_context": None,
                "custom_instructions": custom_instructions,
            }
            
            for stream_key, chunk in graph.stream(
                initial_state,
                config={**checkpoint_config, "recursion_limit": 30},
                stream_mode=["updates", "custom"],
            ):
                if stream_key == "custom":
                    step_data = chunk if isinstance(chunk, dict) else {}
                    if step_data.get("type") == "graph_step":
                        yield ResponsesAgentStreamEvent(
                            type="response.graph_step",
                            item=step_data,
                        )
                    elif step_data.get("type") == "suggested_questions":
                        yield ResponsesAgentStreamEvent(
                            type="response.suggested_questions",
                            item=step_data,
                        )
                    elif step_data.get("type") == "genie_truncated":
                        yield ResponsesAgentStreamEvent(
                            type="response.genie_truncated",
                            item=step_data,
                        )
                    elif step_data.get("type") == "genie_message_ids":
                        if request.custom_inputs is not None and step_data.get("conversation_id") and step_data.get("message_id"):
                            request.custom_inputs["genie_conversation_id"] = step_data["conversation_id"]
                            request.custom_inputs["genie_message_id"] = step_data["message_id"]
                        yield ResponsesAgentStreamEvent(
                            type="response.genie_message_ids",
                            item=step_data,
                        )
                    continue

                # updates: chunk peut être (node_name, state_update) ou {node_name: state_update}
                if isinstance(chunk, (list, tuple)) and len(chunk) == 2:
                    events = {chunk[0]: chunk[1]}
                else:
                    events = chunk

                new_msgs = [
                    msg
                    for v in events.values()
                    if v is not None
                    for msg in v.get("messages", [])
                    if msg.id not in seen_ids
                ]
                
                if first_message:
                    node_name = tuple(events.keys())[0]
                    node_key = node_name[0] if isinstance(node_name, tuple) else node_name
                    if node_key != "simple_genie":
                        seen_ids.update(msg.id for msg in new_msgs[: len(cc_msgs)])
                        new_msgs = new_msgs[len(cc_msgs) :]
                    first_message = False

                node_name = tuple(events.keys())[0]
                if node_name in HIDDEN_NODES:
                    continue

                node_key = node_name[0] if isinstance(node_name, tuple) else node_name
                if node_key == "genie_worker":
                    new_msgs = [msg for v in events.values() for msg in v.get("messages", [])]

                visible_msgs = [m for m in new_msgs if _should_emit(m)]
                for m in visible_msgs:
                    # if hasattr(m, "name") and m.name == "query_response_text" and m.content:
                    #     m.content = f"<genie-text>{m.content}</genie-text>"
                    if hasattr(m, "name") and m.name == "query_sql" and m.content:
                        m.content = f"<genie-sql>{m.content}</genie-sql>"
                    if hasattr(m, "name") and m.name == "query_result" and m.content:
                        m.content = f"<genie-table>{m.content}</genie-table>"
                if visible_msgs:
                    yield from output_to_responses_items_stream(visible_msgs)
                accumulated_output_messages.extend(new_msgs)
                seen_ids.update(msg.id for msg in new_msgs)

#######################################################
# Configuration (multi-persona: Stock, Cdistrib)
#######################################################

import os

LLM_ENDPOINT_NAME = "databricks-claude-opus-4-5"
LLM_FAST_ENDPOINT_NAME = "databricks-claude-opus-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
llm_fast = ChatDatabricks(endpoint=LLM_FAST_ENDPOINT_NAME)

GENIE_CONFIG_STOCK = Genie(
    space_id="XXXXX",
    name="Stock",
    description="Ce Genie Space centralise l'intelligence opérationnelle et technique de la Maison. Il réconcilie la vision logistique (flux et stocks) avec l'ADN technique des produits (nomenclatures, référentiels) et leur réalité économique (coûts, collections).",
)

GENIE_CONFIG_CDISTRIB = Genie(
    space_id= "XXXXXX",
    name="Cdistrib",
    description="Ce Genie Space couvre l'activité du Centre de Distribution : réceptions, expéditions, KPIs leadtime, CQ en cours et réalisés, SLA GDC.",
)

EXTERNALLY_SERVED_AGENTS = [GENIE_CONFIG_STOCK, GENIE_CONFIG_CDISTRIB]

def _load_semantic_context_for_persona(persona_id: str) -> str:
    """Charge le contexte sémantique depuis artifacts ou /tmp ou env."""
    fname = f"semantic_context_{persona_id.lower()}.txt"
    _path = os.path.abspath(__file__)
    for _ in range(3):
        _dir = os.path.dirname(_path)
        _ctx_path = os.path.join(_dir, "artifacts", fname)
        if os.path.exists(_ctx_path):
            with open(_ctx_path, "r", encoding="utf-8") as f:
                return f.read()
        _path = _dir
    _env_key = f"SEMANTIC_CONTEXT_{persona_id.upper()}"
    return os.environ.get(_env_key, "")

SEMANTIC_CONTEXT_STOCK = _load_semantic_context_for_persona("Stock")
SEMANTIC_CONTEXT_CDISTRIB = _load_semantic_context_for_persona("Cdistrib")

PERSONAS = {
    "Stock": (GENIE_CONFIG_STOCK, SEMANTIC_CONTEXT_STOCK),
    "Cdistrib": (GENIE_CONFIG_CDISTRIB, SEMANTIC_CONTEXT_CDISTRIB),
}
DEFAULT_PERSONA = "Stock"

mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(
    lakebase_instance_name=LAKEBASE_INSTANCE_NAME,
    llm=llm,
    personas=PERSONAS,
    default_persona=DEFAULT_PERSONA,
    llm_fast=llm_fast,
)
mlflow.models.set_model(AGENT)

### Resetting the Python environment 

In [0]:
dbutils.library.restartPython()

### Local tests

In [0]:
from agent import AGENT

result = AGENT.predict({
    "input": [{"role": "user", "content": "Quel est le dernier mouvement de stock ?"}]
})
print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]
genie_conv_id = result.custom_outputs.get("genie_conversation_id")
print("thread_id:", thread_id)
print("genie_conversation_id:", genie_conv_id)

In [0]:
response2 = AGENT.predict({
    "input": [{"role": "user", "content": "À quelle heure était ce mouvement ?"}],
    "custom_inputs": {"thread_id": thread_id, "genie_conversation_id": genie_conv_id}
})
print("Response 2:", response2.model_dump(exclude_none=True))

## Logging the agent as an MLflow model

Logs the agent as code from the `agent.py` file. [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enabling automatic authentication for Databricks resources
This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import EXTERNALLY_SERVED_AGENTS, LLM_ENDPOINT_NAME, TOOLS, Genie, LAKEBASE_INSTANCE_NAME
from databricks_langchain import UnityCatalogTool, VectorSearchRetrieverTool
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
    DatabricksTable,
    DatabricksServingEndpoint, 
    DatabricksLakebase
)
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME), DatabricksLakebase(database_instance_name=LAKEBASE_INSTANCE_NAME)]

resources.append(DatabricksSQLWarehouse(warehouse_id="c5669ff49b9955a5"))

resources.append(DatabricksTable(table_name="fshdh_poc_runwai.bills_of_materials.dm_bills_of_materials"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.collection_referential.dm_collection"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.component_referential.dm_base_component"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.component_referential.dm_color_component"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.component_referential.dm_component"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.cost_prices_materials_and_components.ft_component_cost_price"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.finished_product_referential.dm_product_histo"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.finished_product_referential.dm_product_sku"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_flows_materials_and_components.ft_flow_components_detail"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_flows_materials_and_components.ft_flow_components_global"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_flows_materials_and_components.ft_stock_components"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.components_global_buys_orders.ft_oa_line"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.components_global_buys_orders.dm_oa_header"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.fabrication_orders.ft_of_line"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.fabrication_orders.dm_of_header"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.label_tables.lab_storage_labels"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.fashion_icone.ft_global_kpi_leadtime"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.dm_calendar_gdc"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.dm_product_ma"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.dm_epc"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.dm_sla_gdc_leadtime"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.ft_cq_ongoing"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.ft_cq_realized"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.ft_expedition"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.ft_kpi_detail_epc"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.global_dc_activity_finished_products.ft_reception"))
resources.append(DatabricksTable(table_name="fshdh_poc_runwai.label_tables.types_mouvement_orli"))



# Add tools from Unity Catalog
for tool in TOOLS:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is an LLM agent?"
        }
    ],
    "custom_inputs": {"thread_id": "example-thread-123"},
}

# Add serving endpoints and Genie Spaces
for agent in EXTERNALLY_SERVED_AGENTS:
    if isinstance(agent, Genie):
        resources.append(DatabricksGenieSpace(genie_space_id=agent.space_id))
    else:
        resources.append(DatabricksServingEndpoint(endpoint_name=agent.endpoint_name))

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,
        input_example=input_example,
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
            f"databricks-langchain[memory]=={get_distribution('databricks-langchain[memory]').version}",
        ],
        artifacts = { "artifacts/semantic_context_stock.txt": "/tmp/semantic_context_stock.txt", "artifacts/semantic_context_cdistrib.txt": "/tmp/semantic_context_cdistrib.txt" },
    )

## Registering the model in the Unity Catalog

Updating of the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

catalog = "fshdh_poc_runwai"
schema = "ai"
model_name = "agent-genie-with-memory"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

## Deployment of the agent

In [0]:
from databricks import agents

agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags={"endpointSource": "docs"}, deploy_feedback_model=False)